# Module 10 — Snapshot Storage Growth: Why Repo Size Can Exceed Ingested Data

It's a common surprise: you ingested 100 MB of data, but the snapshot repository on disk
is now 800 MB. Compression and segment-level deduplication explain the *steady-state*
size — but they do **not** explain why the repo grows *more than* the data you ingested.

This notebook proves the actual mechanism with measurable before/after numbers.

## Mental model

1. **Lucene segments are immutable.** An update is a `delete + insert`. A delete is a
   tombstone, not a free. Space is only reclaimed when segments are merged.
2. **Background merging** consolidates small segments into larger ones. The old
   segment files are removed from the live index — but if any snapshot already
   captured them, those files were *already* copied to the repo.
3. **Snapshots are file-level incremental.** A snapshot stores references to the
   segment files that exist on disk at snapshot time. Two snapshots that share
   segments share the underlying files in the repo.
4. **Retention is the multiplier.** A file in the repo is deleted only when *no*
   remaining snapshot references it. Long retention + high churn = repo grows
   roughly with `churn × retention`, not with logical data size.

## Scenarios this notebook proves

| # | Scenario | Why repo grows |
|---|---|---|
| A | Document updates | New segments alongside old, until merge |
| B | Document deletes | Tombstones, no space reclaimed until merge |
| C | Background segment merge | Old pre-merge segments still pinned by older snapshots |
| D | **Force-merge** (the operational footgun) | Whole index rewritten; old segments still pinned |
| E | High refresh rate | Many small segments → larger snapshot file lists |
| F | Reindex into a new index | New index has entirely new segment files |
| G | Retention is the multiplier | Deleting old snapshots actually shrinks the repo |

Module 02 covered *what* is in a snapshot; this module covers *how snapshot storage evolves over time*.


---
## 1. Setup

Import helpers, get an Elasticsearch client, and register a **dedicated** filesystem
repository at `snapshot-repo/growth_demo/` so disk measurements aren't polluted by
existing snapshots from earlier modules.


In [ ]:
import os
import time
from helpers import *
import matplotlib.pyplot as plt
from elasticsearch.helpers import bulk

es = get_client()
wait_for_green(es)

In [ ]:
# Dedicated repo for this module — isolated subdirectory under the bind mount.
GROWTH_REPO = "growth_demo"
GROWTH_REPO_SUBPATH = "growth_demo"
GROWTH_INDEX = "growth-demo"

# Ensure host-side directory exists; ES will create files inside it.
host_path = SNAPSHOT_REPO_HOST_PATH / GROWTH_REPO_SUBPATH
host_path.mkdir(parents=True, exist_ok=True)

es.snapshot.create_repository(
    name=GROWTH_REPO,
    body={
        "type": "fs",
        "settings": {
            "location": f"/usr/share/elasticsearch/snapshots/{GROWTH_REPO_SUBPATH}",
            "compress": True,
        },
    },
)
success(f"Repository '{GROWTH_REPO}' registered at subpath '{GROWTH_REPO_SUBPATH}'")
info(f"Host-side path for disk measurement: {host_path}")

In [ ]:
# Growth log — each scenario appends one row that feeds the final tables and charts.
growth_log = []


def index_size_bytes(*indices):
    total = 0
    for idx in indices:
        try:
            s = es.indices.stats(index=idx)
            total += s["_all"]["primaries"]["store"]["size_in_bytes"]
        except Exception:
            pass
    return total


def take_snapshot(snap_name, indices):
    """Create snapshot, wait for completion, return state dict."""
    delete_snapshot_if_exists(es, GROWTH_REPO, snap_name)
    es.snapshot.create(
        repository=GROWTH_REPO,
        snapshot=snap_name,
        body={"indices": ",".join(indices), "include_global_state": False},
        wait_for_completion=False,
    )
    return wait_for_snapshot(es, GROWTH_REPO, snap_name, timeout=600)


def record(label, snap_name, indices=None):
    """Snapshot the given indices, measure everything, append a row to growth_log."""
    if indices is None:
        indices = [GROWTH_INDEX]
    es.indices.refresh(index=",".join(indices))
    snap_info = take_snapshot(snap_name, indices)
    breakdown = snapshot_size_breakdown(es, GROWTH_REPO, snap_name)
    seg = segment_summary(es, ",".join(indices))
    repo_bytes = repo_disk_size_bytes(GROWTH_REPO_SUBPATH)
    idx_bytes = index_size_bytes(*indices)

    row = {
        "label": label,
        "snap": snap_name,
        "repo_disk_bytes": repo_bytes,
        "index_bytes": idx_bytes,
        "snap_total_bytes": breakdown["total_bytes"],
        "snap_incremental_bytes": breakdown["incremental_bytes"],
        "snap_total_files": breakdown["total_files"],
        "snap_incremental_files": breakdown["incremental_files"],
        "num_segments": seg["num_segments"],
        "num_docs": seg["num_docs"],
        "deleted_docs": seg["deleted_docs"],
    }
    growth_log.append(row)

    t = Table(title=f"[bold]{label}[/bold]  ({snap_name})", show_header=True)
    t.add_column("Metric", style="cyan")
    t.add_column("Value", justify="right")
    t.add_row("Repo disk (du-equivalent)", human_bytes(repo_bytes))
    t.add_row("Live index size",           human_bytes(idx_bytes))
    t.add_row("Snapshot total",            human_bytes(breakdown["total_bytes"]))
    t.add_row("Snapshot incremental",      human_bytes(breakdown["incremental_bytes"]))
    t.add_row("Snapshot files (total)",    str(breakdown["total_files"]))
    t.add_row("Snapshot files (this snap)", str(breakdown["incremental_files"]))
    t.add_row("Segments",                  str(seg["num_segments"]))
    t.add_row("Live docs",                 f"{seg['num_docs']:,}")
    t.add_row("Deleted docs (tombstones)", f"{seg['deleted_docs']:,}")
    console.print(t)
    return row

---
## 2. Baseline — fresh ingest

Create a single-shard index and bulk-ingest 500 000 synthetic documents
(~200 B each, so the index should land near ~100 MB). Take the first snapshot.

**Expected:** repo disk ≈ index size. This is our reference point — every later
delta is measured against this.


In [ ]:
# Clean slate (re-runnable)
for idx in [GROWTH_INDEX, "growth-demo-v2", "growth-churn", "growth-bulk"]:
    try:
        es.indices.delete(index=idx)
    except Exception:
        pass

es.indices.create(
    index=GROWTH_INDEX,
    body={
        "settings": {
            "number_of_shards": 1,
            "number_of_replicas": 0,
            "refresh_interval": "30s",
        },
        "mappings": {
            "properties": {
                "id":       {"type": "long"},
                "category": {"type": "keyword"},
                "payload":  {"type": "keyword"},
                "ts":       {"type": "date"},
            }
        },
    },
)
success(f"Index '{GROWTH_INDEX}' created")

In [ ]:
# Bulk-ingest 500k synthetic documents (~200 bytes each on the wire).
N_DOCS = 500_000
PAYLOAD = "x" * 180  # plus a few small fields → ~200 B doc

def doc_gen(n, index, payload):
    for i in range(n):
        yield {
            "_op_type": "index",
            "_index": index,
            "_id": str(i),
            "_source": {
                "id": i,
                "category": f"cat-{i % 50}",
                "payload": payload,
                "ts": "2026-05-12T00:00:00Z",
            },
        }

t0 = time.time()
ok, errors = bulk(
    es,
    doc_gen(N_DOCS, GROWTH_INDEX, PAYLOAD),
    chunk_size=5000,
    request_timeout=120,
    raise_on_error=False,
)
es.indices.refresh(index=GROWTH_INDEX)
success(f"Ingested {ok:,} docs in {time.time()-t0:.1f}s  (errors: {len(errors) if isinstance(errors, list) else errors})")

In [ ]:
record("baseline", "snap-00-baseline")

---
## 3. Scenario A — Document updates

Update the `payload` field on ~50% of documents. In Lucene terms this is a
*delete + insert*: the original docs are tombstoned, new versions are written
into fresh segments. The live index size barely changes — but the **repo grows**
because `snap-00-baseline` still references the old segment files, and the new
snapshot uploads the new segments alongside.


In [ ]:
# Rewrite the payload on docs 0..249,999 (half the index)
result = es.update_by_query(
    index=GROWTH_INDEX,
    body={
        "query": {"range": {"id": {"lt": 250_000}}},
        "script": {"source": "ctx._source.payload = ctx._source.payload + 'U'"},
    },
    conflicts="proceed",
    refresh=True,
    wait_for_completion=True,
    request_timeout=600,
)
success(f"Updated {result['updated']:,} docs in {result['took']/1000:.1f}s")

In [ ]:
record("after-updates", "snap-01-updates")

**Read the table above:** the *Live index size* moved only slightly (same logical data),
but *Repo disk* grew substantially. *Snapshot incremental* shows the actual bytes added
to the repo by this snapshot — almost the size of the rewritten data. The old version
of each updated doc is still on disk, pinned by `snap-00-baseline`.


---
## 4. Scenario B — Document deletes

Delete docs 250 000..399 999 (~30% of the index). These become tombstones in
their segments; no space is reclaimed until a merge. Watch the *Deleted docs*
counter in the next table.


In [ ]:
result = es.delete_by_query(
    index=GROWTH_INDEX,
    body={"query": {"range": {"id": {"gte": 250_000, "lt": 400_000}}}},
    conflicts="proceed",
    refresh=True,
    wait_for_completion=True,
    request_timeout=600,
)
success(f"Deleted {result['deleted']:,} docs in {result['took']/1000:.1f}s")

In [ ]:
record("after-deletes", "snap-02-deletes")

**What changed:** *Deleted docs* jumped to ~150 000 — those tombstones occupy
segment space until merged. *Repo disk* grew again, because the snapshot needed
to capture the updated segment metadata that records the tombstones.


---
## 5. Scenario C — Background segment merge

Lucene's `TieredMergePolicy` continuously merges small segments into larger ones.
The pre-merge segments are removed from the live index — but earlier snapshots
already captured those files in the repo.

To make the effect deterministic we (1) drop `refresh_interval` to create more
small segments, (2) write a small additional batch, and (3) request a partial
merge to roughly halve the segment count. This simulates what background merging
does naturally over time.


In [ ]:
es.indices.put_settings(index=GROWTH_INDEX, body={"index": {"refresh_interval": "500ms"}})

# Small additional write workload — many small segments due to fast refresh.
def small_gen(n, start_id, index, payload):
    for i in range(n):
        yield {
            "_op_type": "index",
            "_index": index,
            "_id": str(start_id + i),
            "_source": {
                "id": start_id + i,
                "category": f"cat-{(start_id + i) % 50}",
                "payload": payload,
                "ts": "2026-05-12T01:00:00Z",
            },
        }

bulk(es, small_gen(10_000, 1_000_000, GROWTH_INDEX, PAYLOAD), chunk_size=500)
es.indices.refresh(index=GROWTH_INDEX)

current_segs = segment_summary(es, GROWTH_INDEX)["num_segments"]
target = max(1, current_segs // 2)
info(f"Current segments: {current_segs}  → merging down to {target}")
es.indices.forcemerge(index=GROWTH_INDEX, max_num_segments=target)
success("Partial merge complete")

In [ ]:
record("after-bg-merge", "snap-03-bg-merge")

**Read the table above:** *Segments* dropped (consolidation worked), but
*Repo disk* and *Snapshot incremental* went up. The merged segments are *new*
files in the repo, and the pre-merge segments are still pinned by `snap-00`..`snap-02`.


---
## 6. Scenario D — Force-merge to 1 segment (the operational footgun)

This is the scenario operators get bitten by most often. After several snapshots
have been taken, running `_forcemerge?max_num_segments=1` rewrites the entire
index into a single new segment. Every previously-snapshotted segment is now
"old" in the index but **still pinned by every prior snapshot in the repo**.

The next snapshot must upload an entirely new copy of all the data. The repo
roughly **doubles** until those older snapshots are deleted.


In [ ]:
es.indices.forcemerge(index=GROWTH_INDEX, max_num_segments=1)
success("Force-merge to 1 segment complete")

In [ ]:
record("after-force-merge", "snap-04-forcemerge")

**Read the table above:** *Segments* is now 1, *Deleted docs* is 0 (tombstones
purged by the merge), *Live index size* is smaller than before — but *Repo disk*
took a sharp jump upward. The next section proves these old bytes are pinned
by the older snapshots, not by anything in the current index.


---
## 7. Scenario E — High refresh rate creates many small segments

Two new indices, same total docs, different refresh behavior:

- `growth-bulk`  — `refresh_interval: 30s` (default-ish, batched)
- `growth-churn` — `refresh_interval: 200ms` (very chatty)

Same data; different segment counts; different snapshot file-list size.
This is measured separately so it doesn't pollute the main growth chart.


In [ ]:
for idx, refresh in [("growth-bulk", "30s"), ("growth-churn", "200ms")]:
    try:
        es.indices.delete(index=idx)
    except Exception:
        pass
    es.indices.create(
        index=idx,
        body={
            "settings": {
                "number_of_shards": 1,
                "number_of_replicas": 0,
                "refresh_interval": refresh,
            },
            "mappings": {"properties": {
                "id": {"type": "long"}, "payload": {"type": "keyword"},
            }},
        },
    )

N_E = 100_000
def churn_gen(n, idx):
    for i in range(n):
        yield {"_op_type": "index", "_index": idx, "_id": str(i),
               "_source": {"id": i, "payload": PAYLOAD}}

# Bulk-mode: large batches.
bulk(es, churn_gen(N_E, "growth-bulk"), chunk_size=5000)

# Churn-mode: small batches with sleeps to force refreshes between them.
batch = []
for i, doc in enumerate(churn_gen(N_E, "growth-churn")):
    batch.append(doc)
    if len(batch) == 1000:
        bulk(es, batch)
        batch = []
        if i % 10_000 == 0:
            time.sleep(0.3)  # let refresh tick over
if batch:
    bulk(es, batch)

es.indices.refresh(index="growth-bulk,growth-churn")
success("Both side-by-side indices populated")

In [ ]:
# Snapshot each separately so we can read their stats independently.
delete_snapshot_if_exists(es, GROWTH_REPO, "snap-bulk-refresh")
delete_snapshot_if_exists(es, GROWTH_REPO, "snap-churn-refresh")

take_snapshot("snap-bulk-refresh", ["growth-bulk"])
take_snapshot("snap-churn-refresh", ["growth-churn"])

t = Table("Index", "Refresh", "Segments", "Index size", "Snap total", "Snap files")
for idx, snap, refresh in [
    ("growth-bulk", "snap-bulk-refresh", "30s"),
    ("growth-churn", "snap-churn-refresh", "200ms"),
]:
    seg = segment_summary(es, idx)
    bd  = snapshot_size_breakdown(es, GROWTH_REPO, snap)
    t.add_row(
        idx, refresh, str(seg["num_segments"]),
        human_bytes(index_size_bytes(idx)),
        human_bytes(bd["total_bytes"]),
        str(bd["total_files"]),
    )
console.print(t)

**Read the comparison:** same logical data, but the churn index has many
more segments, a larger snapshot file count, and slightly larger total bytes
(small segments compress less well and have more per-segment overhead). Over
time, this amplifies via Scenarios C and D — more segments means more merge
churn means more "ghost" files in the repo.


---
## 8. Scenario F — Reindex into a new index

Reindexing produces an entirely new index with entirely new segment files.
The next snapshot now includes both indices, and the bytes for `growth-demo-v2`
are net-new to the repo.


In [ ]:
try:
    es.indices.delete(index="growth-demo-v2")
except Exception:
    pass

es.reindex(
    body={
        "source": {"index": GROWTH_INDEX},
        "dest":   {"index": "growth-demo-v2"},
    },
    wait_for_completion=True,
    refresh=True,
    request_timeout=600,
)
es.indices.refresh(index="growth-demo-v2")
success("Reindex complete")

In [ ]:
record("after-reindex", "snap-05-reindex", indices=[GROWTH_INDEX, "growth-demo-v2"])

**Read the table above:** *Live index size* doubled (two indices now), and
*Repo disk* jumped by approximately the size of the v2 index. None of the v2's
segment files existed in any earlier snapshot, so all of them are incremental.


---
## 9. Scenario G — Retention is the multiplier (proof by deletion)

If older snapshots are what's pinning the bloat, deleting them should make the
repo shrink — even though we don't touch the live index at all.

We delete snapshots one at a time and re-measure disk after each.


In [ ]:
retention_log = [{
    "stage": "before-prune",
    "repo_disk_bytes": repo_disk_size_bytes(GROWTH_REPO_SUBPATH),
}]

for snap in ["snap-00-baseline", "snap-01-updates", "snap-02-deletes", "snap-03-bg-merge"]:
    try:
        es.snapshot.delete(repository=GROWTH_REPO, snapshot=snap)
        info(f"Deleted {snap}")
    except Exception as e:
        warn(f"Could not delete {snap}: {e}")
    time.sleep(1)  # repo housekeeping is async
    retention_log.append({
        "stage": f"after-delete-{snap}",
        "repo_disk_bytes": repo_disk_size_bytes(GROWTH_REPO_SUBPATH),
    })

t = Table("Stage", "Repo disk", "Δ vs before-prune")
baseline_disk = retention_log[0]["repo_disk_bytes"]
for r in retention_log:
    delta = r["repo_disk_bytes"] - baseline_disk
    t.add_row(r["stage"], human_bytes(r["repo_disk_bytes"]),
              f"{'+' if delta >= 0 else ''}{human_bytes(delta)}")
console.print(t)

**Punchline:** the live index never changed during this section. The only
thing we did was prune older snapshots, and the repo gave back the bytes those
snapshots were pinning. The repo's true steady-state size is governed by the
retention policy and the churn between retained snapshots — not by the size of
the live data.


---
## 10. Summary

A single table and two charts that tell the story.


In [ ]:
t = Table(title="Snapshot storage growth across scenarios", show_header=True)
t.add_column("Scenario", style="cyan")
t.add_column("Snapshot")
t.add_column("Repo disk", justify="right")
t.add_column("Δ vs baseline", justify="right")
t.add_column("Index size", justify="right")
t.add_column("Snap total", justify="right")
t.add_column("Snap incr.", justify="right")
t.add_column("Segs", justify="right")
t.add_column("Tombstones", justify="right")

baseline_disk = growth_log[0]["repo_disk_bytes"] if growth_log else 0
for r in growth_log:
    delta = r["repo_disk_bytes"] - baseline_disk
    t.add_row(
        r["label"],
        r["snap"],
        human_bytes(r["repo_disk_bytes"]),
        f"{'+' if delta >= 0 else ''}{human_bytes(delta)}",
        human_bytes(r["index_bytes"]),
        human_bytes(r["snap_total_bytes"]),
        human_bytes(r["snap_incremental_bytes"]),
        str(r["num_segments"]),
        f"{r['deleted_docs']:,}",
    )
console.print(t)

In [ ]:
# Chart 1: repo disk vs live index size across scenarios
labels = [r["label"] for r in growth_log]
repo_mb = [r["repo_disk_bytes"] / 1024 / 1024 for r in growth_log]
idx_mb  = [r["index_bytes"]      / 1024 / 1024 for r in growth_log]
incr_mb = [r["snap_incremental_bytes"] / 1024 / 1024 for r in growth_log]

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(labels, repo_mb, marker="o", linewidth=2, label="Repo disk (du)")
ax.plot(labels, idx_mb,  marker="s", linewidth=2, label="Live index size")
ax.bar(labels, incr_mb, alpha=0.25, label="Snapshot incremental upload")
ax.set_ylabel("Size (MB)")
ax.set_title("Snapshot repo disk grows even when live data does not")
ax.legend(loc="upper left")
ax.grid(True, alpha=0.3)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Chart 2: repo disk shrinking as old snapshots are pruned
r_labels = [r["stage"] for r in retention_log]
r_mb     = [r["repo_disk_bytes"] / 1024 / 1024 for r in retention_log]

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(r_labels, r_mb, marker="o", linewidth=2, color="C3")
ax.set_ylabel("Repo disk (MB)")
ax.set_title("Deleting old snapshots actually shrinks the repo")
ax.grid(True, alpha=0.3)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

---
## 11. Mitigation cheat sheet

| Pattern | Why it helps |
|---|---|
| Snapshot **before** force-merging, never after — or accept the doubling cost | The post-merge snapshot has to re-upload the entire index; the pre-merge snapshot remains valid and useful. |
| Tune SLM retention to the smallest viable window | Retention is the linear multiplier on churn → repo size. |
| Prefer **data streams + ILM** for append-only data (see module 08) | Rollover-based indices are immutable once rolled over, eliminating the in-place-mutation source of pinned segments. |
| Use **source-only repositories** for cold archival | Stores only `_source`, no doc values/postings → far smaller, restore-into-new-cluster only. |
| Consider **searchable snapshots** for archival tiers (see module 07) | Data lives in the snapshot repo; the cluster keeps no local copy. |
| Avoid frequent updates/deletes on snapshot-tracked indices when possible | The cheapest way to keep repos small is to write data that doesn't churn. |
| Monitor repo disk separately from index disk | They diverge over time; this notebook shows by how much. |


---
## 12. Cleanup

Make this notebook safely re-runnable: delete remaining snapshots, the repo,
and the demo indices.


In [ ]:
# Delete any snapshots that survived Section 9.
for snap in ["snap-04-forcemerge", "snap-05-reindex",
             "snap-bulk-refresh", "snap-churn-refresh"]:
    delete_snapshot_if_exists(es, GROWTH_REPO, snap)

# Drop the repo (this removes its registration; files on disk under the
# location may remain — Elasticsearch only deletes files it created via
# the snapshot API while the repo is registered).
try:
    es.snapshot.delete_repository(name=GROWTH_REPO)
    success(f"Repository '{GROWTH_REPO}' deleted")
except Exception as e:
    warn(f"Could not delete repository: {e}")

for idx in [GROWTH_INDEX, "growth-demo-v2", "growth-churn", "growth-bulk"]:
    try:
        es.indices.delete(index=idx)
        info(f"Deleted index '{idx}'")
    except Exception:
        pass

success("Cleanup complete — notebook is ready to re-run.")